# Lambda: LLM-based Cell Type Annotation

This tutorial demonstrates how to use Lambda for automatic cell type annotation using large language models.

Lambda annotates cell clusters without requiring reference datasets by leveraging LLMs to propose cell type names based on differentially expressed marker genes.

## Key Features:
- No reference dataset required
- Uses LLMs (OpenAI, Google) for annotation
- Multi-round reasoning for improved accuracy
- Supports hierarchical annotations

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Import spatialvi
import spatialvi
from spatialvi.external import Lambda

sc.set_figure_params(figsize=(6, 6))
print("spatialvi version:", spatialvi.__version__)

## Prerequisites

Lambda requires:
1. The `LAMBDA` package: `pip install LAMBDA`
2. An API key for OpenAI or Google set as environment variable:
   - `OPENAI_API_KEY` for OpenAI
   - `GOOGLE_API_KEY` for Google

In [ ]:
# Check for API key (optional)
import os

if "OPENAI_API_KEY" in os.environ:
    print("OpenAI API key found")
elif "GOOGLE_API_KEY" in os.environ:
    print("Google API key found")
else:
    print("Warning: No API key found. Set OPENAI_API_KEY or GOOGLE_API_KEY")

## 1. Load and Preprocess Data

In [ ]:
# Load example spatial data
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()

print(adata)

In [ ]:
# Preprocessing
sc.pp.filter_genes(adata, min_cells=10)

# Normalize and log transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Select highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

print(f"Preprocessed data: {adata.n_obs} spots, {adata.n_vars} genes")

In [ ]:
# Cluster the data (Lambda can also do this internally)
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.leiden(adata, resolution=0.5)

print("\nCluster distribution:")
print(adata.obs["leiden"].value_counts())

In [ ]:
# Visualize clusters
sc.tl.umap(adata)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="leiden", ax=axes[0], show=False, title="Clusters (UMAP)")
sc.pl.spatial(adata, color="leiden", spot_size=80, ax=axes[1], show=False, title="Clusters (Spatial)")

plt.tight_layout()
plt.show()

## 2. Find Marker Genes

Lambda uses differentially expressed genes to identify cell types.

In [ ]:
# Find marker genes for each cluster
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

# Display top markers
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)

In [ ]:
# Get marker genes DataFrame
markers = sc.get.rank_genes_groups_df(adata, group=None)

print("Top 5 marker genes per cluster:")
for cluster in adata.obs["leiden"].cat.categories[:5]:
    cluster_markers = markers[markers["group"] == cluster].head(5)["names"].tolist()
    print(f"  Cluster {cluster}: {', '.join(cluster_markers)}")

## 3. Initialize Lambda Model

In [ ]:
# Initialize Lambda model
model = Lambda(
    adata,
    location="lymph node",  # Anatomical location
    organism="human",  # Species
    provider="openai",  # LLM provider (openai or google)
    num_parallel=10,  # Concurrent annotations
    n_top_genes=50,  # Number of DE genes to use
    resolution=0.5,  # Leiden resolution if clustering needed
)

print("Lambda model initialized")

## 4. Prepare the Agent

In [ ]:
# Initialize the LAMBDA agent
try:
    model.train()
    print("LAMBDA agent initialized successfully")
except ImportError as e:
    print(f"Note: {e}")
    print("Install with: pip install LAMBDA")

## 5. Annotate Clusters

In [ ]:
# Run annotation
try:
    adata = model.predict(
        group_key="leiden",  # Key in obs for clusters
        groups=None,  # Annotate all groups
        key_is_hierarchical=False,
        level=0,
        store_key_prefix="lambda",
    )
    print("Annotation complete!")

    # Show results
    if "lambda_0" in adata.obs.columns:
        print("\nCell type annotations:")
        print(adata.obs["lambda_0"].value_counts())
except (ImportError, RuntimeError) as e:
    print(f"Note: Cannot run annotation - {e}")
    print("Creating example annotations for demonstration...")

    # Create example annotations
    example_annotations = {
        "0": "B cells",
        "1": "T cells",
        "2": "Macrophages",
        "3": "Dendritic cells",
        "4": "Fibroblasts",
        "5": "Endothelial cells",
        "6": "Plasma cells",
        "7": "NK cells",
    }
    adata.obs["lambda_0"] = adata.obs["leiden"].map(lambda x: example_annotations.get(x, "Unknown"))
    adata.obs["lambda_score_0"] = np.random.uniform(0.7, 0.95, adata.n_obs)

## 6. Visualize Annotations

In [ ]:
# Visualize annotations
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.umap(adata, color="lambda_0", ax=axes[0], show=False, title="Cell Type Annotations (UMAP)")
sc.pl.spatial(adata, color="lambda_0", spot_size=80, ax=axes[1], show=False, title="Cell Type Annotations (Spatial)")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize annotation confidence scores
if "lambda_score_0" in adata.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Histogram of scores
    axes[0].hist(adata.obs["lambda_score_0"], bins=30, edgecolor="black")
    axes[0].set_xlabel("Confidence Score")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of Annotation Confidence")

    # Spatial view of confidence
    sc.pl.spatial(
        adata, color="lambda_score_0", spot_size=80, ax=axes[1], show=False, title="Annotation Confidence (Spatial)"
    )

    plt.tight_layout()
    plt.show()

## 7. Compare Annotations with Markers

In [ ]:
# Visualize known marker genes to validate annotations
# Common immune cell markers
marker_genes = {
    "B cells": ["CD19", "CD79A", "MS4A1"],
    "T cells": ["CD3D", "CD3E", "CD8A"],
    "Macrophages": ["CD68", "CD163", "CSF1R"],
    "Dendritic cells": ["ITGAX", "CD1C", "CLEC9A"],
}

# Find available markers
available_markers = []
for _cell_type, genes in marker_genes.items():
    for gene in genes:
        if gene in adata.var_names:
            available_markers.append(gene)

if len(available_markers) > 0:
    # Plot available markers
    n_markers = min(len(available_markers), 6)
    sc.pl.spatial(adata, color=available_markers[:n_markers], ncols=3, spot_size=60)
else:
    print("No common marker genes found in dataset")

## 8. Get Annotation Results

In [ ]:
# Get annotations from the model
try:
    annotations = model.get_annotations(level=0)
    print("Annotation results:")
    print(annotations)
except (RuntimeError, AttributeError):
    print("Annotations are stored in adata.obs:")
    print(adata.obs[["leiden", "lambda_0"]].drop_duplicates().sort_values("leiden"))

## 9. Summary Statistics

In [ ]:
# Cell type composition
print("Cell type composition:")
cell_type_counts = adata.obs["lambda_0"].value_counts()
print(cell_type_counts)

# Plot pie chart
fig, ax = plt.subplots(figsize=(10, 8))
cell_type_counts.plot(kind="pie", autopct="%1.1f%%", ax=ax)
ax.set_ylabel("")
ax.set_title("Cell Type Composition")
plt.tight_layout()
plt.show()

In [ ]:
# Mean confidence per cell type
if "lambda_score_0" in adata.obs.columns:
    confidence_by_type = adata.obs.groupby("lambda_0")["lambda_score_0"].mean().sort_values(ascending=False)
    print("\nMean confidence by cell type:")
    for ct, score in confidence_by_type.items():
        print(f"  {ct}: {score:.3f}")

## Summary

In this tutorial, we demonstrated:

1. How to prepare spatial data for Lambda annotation
2. How to cluster data and find marker genes
3. How to initialize the Lambda model with context information
4. How to run automatic cell type annotation
5. How to visualize and validate annotations
6. How to assess annotation confidence

Lambda provides a reference-free approach to cell type annotation using large language models, making it particularly useful when reference datasets are unavailable.